# 05 — N3 reconstruction: 2-bubble and 3-bubble comparisons

For each λ̄ we make **4 comparisons** (3 pair-wise + 1 three-bubble):

- **Panels 1–3**: each 2-bubble sledgehamr run vs reconstruction from scan (1D BM, w=1)
- **Panel 4**: 3-bubble sledgehamr run vs two reconstructions:
  - (A) from 3D 2-bubble snapshots × collision weights
  - (B) from 1D BM scan × collision weights

**Workflow**
1. Run **Section 1** to write the weights inputs and SLURM script.
2. Submit `sbatch scripts/run_weights_N3.slurm` and wait for it to finish.
3. Continue from **Section 2** to load the results and make the plots.

In [ ]:
import glob
import os
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.physics import InstantonProfile, BubbleKinematics
from ptbuilder.analysis import (write_weights_input, load_weights,
                                 bubblemaster_normalization, apply_keffsq)
from ptbuilder.scan import load_scan, load_bm_result
from ptbuilder.ic import _cutoff_times

# Bubble positions — N3_234 (lb=0.84)
pos_084 = np.array([[72.25, 89.92, 100.],
                    [127.8, 89.92, 100.],
                    [108.7, 110.1, 100.]])

# Bubble positions — N3_456 (lb=0.069)
pos_069 = np.array([[16.23, 19.06, 22.5],
                    [28.77, 19.06, 22.5],
                    [24.08, 25.94, 22.5]])

In [ ]:
REPO_ROOT   = Path(pt.__file__).parent.parent
SH_RUNS_DIR = Path('/pscratch/sd/b/buschman/other_runs/')
DATA_DIR    = REPO_ROOT / 'data'

# Scan grids differ between lambda values:
#   lb=0.84  → t ∈ [1, 70],  32 points
#   lb=0.069 → t ∈ [1, 30],  32 points  (shorter collision timescale)
# t_scan_max and n_scan are stored per-lambda in CASES below.
# The weights binary uses a finer, per-lambda time grid derived from the same range.
N_SCAN        = 32     # number of BM time steps (fixed by what was run)
N_WEIGHTS     = 500    # weights time resolution (independent of scan)
SCAN_GAMMAS   = np.linspace(1., 8., N_SCAN)


# Per-lambda configuration.
# t_scan_max: upper end of the BM scan time axis for this lambda.
# Each pair dict carries its own 2-bubble weights_in/out paths.
# d, gamma, t_m are NOT hardcoded — computed on the fly from positions and weights output.
CASES = {
    0.84: dict(
        positions=pos_084,
        L=200., zero_pad=2.,
        t_scan_max=70.,
        pairs=[
            dict(idx=(0,1),
                 sh_path=SH_RUNS_DIR / 'bubble_N3_234_0_1_',
                 weights_in =DATA_DIR / 'weights_in_N3_084_pair_0_1.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_084_pair_0_1.h5'),
            dict(idx=(0,2),
                 sh_path=SH_RUNS_DIR / 'bubble_N3_234_0_2_',
                 weights_in =DATA_DIR / 'weights_in_N3_084_pair_0_2.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_084_pair_0_2.h5'),
            dict(idx=(1,2),
                 sh_path=SH_RUNS_DIR / 'bubble_normalization_gamma2',
                 weights_in =DATA_DIR / 'weights_in_N3_084_pair_1_2.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_084_pair_1_2.h5'),
        ],
        sh_path_3b=SH_RUNS_DIR / 'bubble_N3_234_0_1_2',
        weights_in =DATA_DIR / 'weights_in_N3_084_3b.h5',
        weights_out=DATA_DIR / 'weights_out_N3_084_3b.h5',
    ),
    0.069: dict(
        positions=pos_069,
        L=50., zero_pad=2.,
        t_scan_max=30.,
        pairs=[
            dict(idx=(0,1),
                 sh_path=SH_RUNS_DIR / 'bubble_N3_456_0_1_',
                 weights_in =DATA_DIR / 'weights_in_N3_069_pair_0_1.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_069_pair_0_1.h5'),
            dict(idx=(0,2),
                 sh_path=SH_RUNS_DIR / 'bubble_N3_456_0_2_',
                 weights_in =DATA_DIR / 'weights_in_N3_069_pair_0_2.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_069_pair_0_2.h5'),
            dict(idx=(1,2),
                 sh_path=SH_RUNS_DIR / 'bubble_N3_456_1_2_',
                 weights_in =DATA_DIR / 'weights_in_N3_069_pair_1_2.h5',
                 weights_out=DATA_DIR / 'weights_out_N3_069_pair_1_2.h5'),
        ],
        sh_path_3b=SH_RUNS_DIR / 'bubble_N3_456_0_1_2',
        weights_in =DATA_DIR / 'weights_in_N3_069_3b.h5',
        weights_out=DATA_DIR / 'weights_out_N3_069_3b.h5',
    ),
}

# Derived per-lambda time grids
for lb, case in CASES.items():
    case['scan_times']    = np.linspace(1., case['t_scan_max'], N_SCAN)
    case['weights_times'] = np.linspace(1., case['t_scan_max'], N_WEIGHTS)

## 1. Set up weights calculation

Writes the HDF5 inputs and a SLURM script. Kinematics are loaded from the
stored `instanton.h5`; no compiled Python extension is required.

In [ ]:
def load_kinematics_local(lb):
    """Load BubbleKinematics from stored instanton.h5 (no C++ needed)."""
    path = DATA_DIR / f'phi4_lambda_bar{lb}' / 'instanton.h5'
    with h5py.File(path, 'r') as f:
        profile = InstantonProfile(
            R=f['R'][:], Phi=f['Phi'][:],
            rin_0 =float(f.attrs['rin_0']),
            rmid_0=float(f.attrs['rmid_0']),
            rout_0=float(f.attrs['rout_0']),
            interp=None,
        )
    return BubbleKinematics(profile)


weights_dir = REPO_ROOT / 'cpp/weights'
weights_bin = weights_dir / 'weights'
scripts_dir = REPO_ROOT / 'scripts'
logs_dir = REPO_ROOT / 'logs'
scripts_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

commands = []

for lb, case in CASES.items():
    kin      = load_kinematics_local(lb)
    wt       = case['weights_times']   # per-lambda fine time grid
    commands.append(f'# lambda_bar={lb}')

    for pair in case['pairs']:
        i, j     = pair['idx']
        pos_pair = case['positions'][[i, j]]
        write_weights_input(
            path=pair['weights_in'],
            positions=pos_pair,
            t=wt,
            kinematics=kin,
            L=case['L'],
        )
        print(f'Written: {pair["weights_in"].name}')
        commands.append(
            f'srun -n 1 -c 128 {weights_bin} {pair["weights_in"]} {pair["weights_out"]}')

    write_weights_input(
        path=case['weights_in'],
        positions=case['positions'],
        t=wt,
        kinematics=kin,
        L=case['L'],
    )
    print(f'Written: {case["weights_in"].name}')
    commands += [
        f'srun -n 1 -c 128 {weights_bin} {case["weights_in"]} {case["weights_out"]}',
        '',
    ]

script_path = scripts_dir / 'run_weights_N3.slurm'
script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=weights_N3
#SBATCH --output={logs_dir}/weights_N3_%j.out
#SBATCH --constraint=cpu
#SBATCH --qos=regular
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=128
#SBATCH --time=00:10:00

set -euo pipefail
make -C {weights_dir}
export OMP_NUM_THREADS=${{SLURM_CPUS_PER_TASK}}
export OMP_PLACES=threads
export OMP_PROC_BIND=spread

''' + '\n'.join(commands))
print(f'Written: {script_path}')
print(f'Submit with:  sbatch {script_path}')

## 2. Load scan results

In [ ]:
scan_results = {}

for lb, case in CASES.items():
    cache = DATA_DIR / f'phi4_lambda_bar{lb}' / 'scan_cache.h5'
    if cache.exists():
        scan_results[lb] = load_scan(cache)
        actual_t = next(iter(scan_results[lb].values())).times
        print(f'lb={lb}: {len(scan_results[lb])} gamma values from cache  '
              f't=[{actual_t[0]:.1f}, {actual_t[-1]:.1f}]  n_t={len(actual_t)}')
        # Sanity-check against CASES config
        if not (np.isclose(actual_t[-1], case['t_scan_max'], rtol=1e-3) and
                len(actual_t) == N_SCAN):
            print(f'  WARNING: expected t_max={case["t_scan_max"]}, n={N_SCAN}')
    else:
        # Fallback: load from per-gamma directories using per-lambda scan times
        results  = {}
        scan_dir = DATA_DIR / f'phi4_lambda_bar{lb}' / 'scan'
        for gdir in sorted(scan_dir.glob('out_gamma*')):
            sg = float(gdir.name.replace('out_gamma', ''))
            try:
                results[sg] = load_bm_result(gdir, sg, case['scan_times'])
            except Exception as e:
                print(f'  Skipping {gdir.name}: {e}')
        scan_results[lb] = results
        print(f'lb={lb}: {len(results)} gamma values from directories')

## 3. Load collision weights

Continue after the Section 1 SLURM job has completed.

In [ ]:
pair_weights = {}   # (lb, idx_tuple) -> (n_weights_t,) weight array
pair_gammas  = {}   # (lb, idx_tuple) -> gamma (scalar, from weights output)
pair_d       = {}   # (lb, idx_tuple) -> centre-to-centre separation (scalar)
pair_tm      = {}   # (lb, idx_tuple) -> BM comparison time t_m (scalar)
trio_weights = {}   # lb -> dict(weights, pair_i, pair_j, gammas)

for lb, case in CASES.items():
    n_wt = len(case['weights_times'])
    L    = case['L']
    pos  = case['positions']
    kin  = load_kinematics_local(lb)

    for pair in case['pairs']:
        if not pair['weights_out'].exists():
            print(f'lb={lb} pair {pair["idx"]}: not found — submit scripts/run_weights_N3.slurm')
            continue
        w, pi, pj, gam = load_weights(pair['weights_out'], n_wt)

        i, j = pair['idx']
        dp = pos[i] - pos[j]
        dp -= L * np.round(dp / L)
        d     = float(np.linalg.norm(dp))
        t_m   = float(_cutoff_times(d, 0, 1.)[2])   # BM comparison time ≈ 1.095*d
        gamma = float(gam[0])

        print(f'lb={lb} pair {pair["idx"]}:  gamma={gamma:.4f}  d={d:.4f}  t_m={t_m:.4f}  '
              f'mean w={w[0].mean():.4f}  [{w[0].min():.4f}, {w[0].max():.4f}]')
        pair_weights[(lb, pair['idx'])] = w[0]
        pair_gammas [(lb, pair['idx'])] = gamma
        pair_d      [(lb, pair['idx'])] = d
        pair_tm     [(lb, pair['idx'])] = t_m

    if not case['weights_out'].exists():
        print(f'lb={lb} 3-bubble weights: not found — submit scripts/run_weights_N3.slurm')
        continue
    w, pi, pj, gam = load_weights(case['weights_out'], n_wt)
    trio_weights[lb] = dict(weights=w, pair_i=pi.astype(int),
                            pair_j=pj.astype(int), gammas=gam)
    print(f'lb={lb} 3-bubble: {len(pi)} pairs')
    for k in range(len(pi)):
        print(f'  pair ({pi[k]},{pj[k]}):  gamma={gam[k]:.4f}  mean w={w[k].mean():.4f}')

## 4. Load sledgehamr 3D spectra

Loads all time snapshots for each run.  keff² correction is applied to the
x-axis (replacing integer k² shell labels with shell-averaged effective k²).

In [ ]:
def _list_gw_files(sh_path):
    pattern = os.path.join(sh_path, 'gw_spectra', '*', 'spectra.hdf5')
    entries = []
    for fp in glob.glob(pattern):
        dname = os.path.basename(os.path.dirname(fp))
        if dname.isdigit():
            entries.append((int(dname), fp))
    entries.sort(key=lambda x: x[0])
    return entries


def _read_gw_file(fp):
    with h5py.File(fp, 'r') as f:
        k_int = np.asarray(f['k' if 'k' in f else 'k_sq'][:])
        spec  = np.asarray(f['Spectrum' if 'Spectrum' in f else 'spectrum'][:])
        hdr   = np.asarray(f['Header'][:]).reshape(-1)
        t     = float(hdr[0])
    return apply_keffsq(k_int), spec, t


def load_sh_all_snaps(sh_path):
    """Load all GW snapshots; returns list of (k_eff_sq, spectrum, t)."""
    files = _list_gw_files(sh_path)
    if not files:
        raise FileNotFoundError(f'No gw_spectra/*/spectra.hdf5 in {sh_path}')
    snaps = [_read_gw_file(fp) for _, fp in files]
    snaps.sort(key=lambda x: x[2])
    return snaps   # list of (k_eff_sq, spectrum, t)


def normalize_sh(k_sq, spectrum, L, zero_pad):
    """Convert raw sledgehamr spectrum to dE/dlnk on physical k axis."""
    if k_sq[0] == 0:
        k_sq     = k_sq[1:]
        spectrum = spectrum[1:]
    Leff = L * zero_pad
    r    = np.sqrt(k_sq)
    k    = r * (2 * np.pi / Leff)   # physical wavenumber
    norm = 4.0 * (2 * np.pi * r) * Leff**3
    return k, spectrum * norm


# Load snapshots for all runs
sh_snaps = {}   # (lb, tag) -> list of (k_eff_sq, spectrum, t)

for lb, case in CASES.items():
    for pair in case['pairs']:
        tag = f'lb{lb}_pair{pair["idx"]}'
        try:
            sh_snaps[(lb, pair['idx'])] = load_sh_all_snaps(pair['sh_path'])
            nt = len(sh_snaps[(lb, pair['idx'])])
            t0 = sh_snaps[(lb, pair['idx'])][0][2]
            t1 = sh_snaps[(lb, pair['idx'])][-1][2]
            print(f'[{tag}]  {nt} snapshots, t=[{t0:.2f}, {t1:.2f}]')
        except Exception as e:
            print(f'[{tag}]  ERROR: {e}')

    tag3b = f'lb{lb}_3b'
    try:
        sh_snaps[(lb, '3b')] = load_sh_all_snaps(case['sh_path_3b'])
        nt = len(sh_snaps[(lb, '3b')])
        t0 = sh_snaps[(lb, '3b')][0][2]
        t1 = sh_snaps[(lb, '3b')][-1][2]
        print(f'[{tag3b}]  {nt} snapshots, t=[{t0:.2f}, {t1:.2f}]')
    except Exception as e:
        print(f'[{tag3b}]  ERROR: {e}')

## 5. Reconstruct and compare

### Reconstruction formula

For each pair (i,j) with collision weight time series w_{ij}(t):

$$S_\mathrm{recon}(k) = \sum_{\text{pairs}} \sum_t w_{ij}(t)\, \Delta E_{ij}(\gamma_{ij},\, t, k)$$

where $\Delta E$ is the increment in cumulative GW energy between consecutive
time steps, taken either from the **1D BM scan** or the **3D snapshots**.

For 2-bubble panels the weights binary is run with N=2 (single pair, no
blocking), so w≈1 throughout — but the code path is identical to the
3-bubble case.

In [ ]:
from scipy.interpolate import RegularGridInterpolator


def build_log_scan_interpolator(scan_results, k_out):
    """Interpolates log(spectrum) over (gamma, t) grid; returns exp on query."""
    gammas = np.array(sorted(scan_results.keys()))
    times  = next(iter(scan_results.values())).times
    Ng, Nt, Nk = len(gammas), len(times), len(k_out)

    spec_grid = np.zeros((Ng, Nt, Nk))
    for ig, gamma in enumerate(gammas):
        res = scan_results[gamma]
        for it in range(Nt):
            spec_grid[ig, it] = np.interp(k_out, res.w, res.spectrum[it],
                                           left=0., right=0.)

    floor     = spec_grid.max() * 1e-12
    log_grid  = np.log(np.maximum(spec_grid, floor))
    log_floor = np.log(floor)

    _raw = RegularGridInterpolator(
        (gammas, times), log_grid,
        bounds_error=False, fill_value=log_floor,
    )

    def interp_fn(pts):
        return np.exp(_raw(pts))

    return interp_fn, gammas, times, k_out


def bm_reconstruct(gamma, weights_t, weights_times, t_max, interp_fn, times, gammas):
    """
    Reconstruct dE/dlnk from the 1D BM scan for one pair.

    Steps at the FINE weights_times resolution, not the coarse (32-point)
    scan grid: the collision weight can vary much faster in time (brief
    collision windows) than the spectrum (smooth, slowly-varying in log), so
    the reconstruction must be time-stepped at the weights' resolution, with
    the spectrum freshly interpolated from the scan at each fine step --
    the scan exists specifically to be interpolated, not to define the
    reconstruction's own time grid. Matches
    reference_codes/notebooks/Scan.ipynb's combined_spectrum loop.
    """
    g = float(np.clip(gamma, gammas.min(), gammas.max()))

    t_hi   = min(t_max, times[-1])
    mask   = (weights_times >= times[0]) & (weights_times <= t_hi)
    t_comp = weights_times[mask]
    if len(t_comp) == 0 or t_comp[-1] < t_hi:
        t_comp = np.append(t_comp, t_hi)

    if weights_t is None:
        w = np.ones(len(t_comp))
    else:
        w = np.interp(t_comp, weights_times, weights_t)

    pts   = np.column_stack([np.full(len(t_comp), g), t_comp])
    specs = interp_fn(pts)                       # (n_t_fine, n_k)

    delta = np.diff(specs, axis=0)               # (n_t_fine-1, n_k)
    return (w[:len(t_comp) - 1, None] * delta).sum(axis=0)


def interp_sh_at_t(snaps, t_target):
    """Log-space interpolation of snapshot list to t_target."""
    ts = np.array([s[2] for s in snaps])
    if t_target <= ts[0]:
        return snaps[0][0], snaps[0][1], ts[0]
    if t_target >= ts[-1]:
        return snaps[-1][0], snaps[-1][1], ts[-1]
    i = int(np.searchsorted(ts, t_target)) - 1
    k_sq, s_a, t_a = snaps[i]
    _,    s_b, t_b = snaps[i + 1]
    alpha = (t_target - t_a) / (t_b - t_a)
    floor = max(s_a.max(), s_b.max()) * 1e-12
    log_a = np.log(np.maximum(s_a, floor))
    log_b = np.log(np.maximum(s_b, floor))
    return k_sq, np.exp(log_a + alpha * (log_b - log_a)), t_target


def sh_reconstruct(lb, pair_idx, weights_t, weights_times, t_max, k_out, L, zero_pad):
    """Reconstruct dE/dlnk from 3D sledgehamr snapshots for one pair."""
    snaps    = sh_snaps[(lb, pair_idx)]
    combined = np.zeros(len(k_out))

    for i in range(len(snaps) - 1):
        k_sq_a, s_a, t_a = snaps[i]
        k_sq_b, s_b, t_b = snaps[i + 1]
        if t_b > t_max:
            break

        k_a, y_a = normalize_sh(k_sq_a, s_a, L, zero_pad)
        k_b, y_b = normalize_sh(k_sq_b, s_b, L, zero_pad)

        ya_r = np.interp(k_out, k_a, y_a, left=0., right=0.)
        yb_r = np.interp(k_out, k_b, y_b, left=0., right=0.)

        t_mid = 0.5 * (t_a + t_b)
        w_mid = (1.0 if weights_t is None
                 else float(np.interp(t_mid, weights_times, weights_t)))

        combined += w_mid * (yb_r - ya_r)

    return combined

In [ ]:
def load_bm_direct(lb, gamma):
    """Load direct BubbleMaster validation run log-interpolated to t_m."""
    tag     = f'lb{lb}_g{gamma:.4g}'
    val_dir = DATA_DIR / f'phi4_lambda_bar{lb}' / 'validation' / tag
    setup_p = val_dir / 'setup.h5'
    out_dir = val_dir / 'out'

    if not setup_p.exists():
        return None

    with h5py.File(setup_p, 'r') as f:
        t_m   = float(f.attrs['t_m'])
        times = f['times'][:]

    result_files = sorted(out_dir.glob('result_*.h5'),
                          key=lambda p: int(p.stem.split('_')[1]))
    if not result_files:
        return None

    t_vals, spectra, w = [], [], None
    for i, fp in enumerate(result_files):
        if i < len(times):
            t_vals.append(times[i])
        with h5py.File(fp, 'r') as f:
            if w is None:
                w = f['w'][:]
            spectra.append(f['spectrum'][:])
    t_vals = np.array(t_vals)

    if t_m <= t_vals[0]:
        spec = spectra[0]
    elif t_m >= t_vals[-1]:
        spec = spectra[-1]
    else:
        i = int(np.searchsorted(t_vals, t_m)) - 1
        alpha = (t_m - t_vals[i]) / (t_vals[i+1] - t_vals[i])
        floor = max(spectra[i].max(), spectra[i+1].max()) * 1e-12
        log_a = np.log(np.maximum(spectra[i],   floor))
        log_b = np.log(np.maximum(spectra[i+1], floor))
        spec  = np.exp(log_a + alpha * (log_b - log_a))

    return w, spec, t_m


bm_direct = {}
for lb, case in CASES.items():
    for pair in case['pairs']:
        key   = (lb, pair['idx'])
        gamma = pair_gammas.get(key)
        if gamma is None:
            print(f'lb={lb} pair {pair["idx"]}: gamma not loaded — run weights first')
            continue
        result = load_bm_direct(lb, gamma)
        if result is not None:
            bm_direct[key] = result
            w, s, tm = result
            print(f'lb={lb} pair {pair["idx"]}  gamma={gamma:.4f}  t_m={tm:.3f}  spec_max={s.max():.1f}')
        else:
            print(f'lb={lb} pair {pair["idx"]}: validation run not found')

In [ ]:
colors_pair = ['#0072B2', '#D55E00', '#009E73']
colors_3b   = '#CC79A7'

for lb, case in CASES.items():
    L        = case['L']
    zero_pad = case['zero_pad']
    wt       = case['weights_times']

    first_res = next(iter(scan_results[lb].values()))
    k_out = first_res.w

    interp_fn, gammas_grid, scan_t, _ = build_log_scan_interpolator(
        scan_results[lb], k_out)

    # t_m_3b = time of last collision = max t_m across all pairs
    t_m_3b = max(pair_tm.get((lb, p['idx']), 0.) for p in case['pairs'])

    fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)

    # ------------------------------------------------------------------ #
    # Panels 1–3: pair-wise comparisons at t_m                           #
    # ------------------------------------------------------------------ #
    for ax, pair, color in zip(axes[:3], case['pairs'], colors_pair):
        key   = (lb, pair['idx'])
        d     = pair_d    .get(key)
        gamma = pair_gammas.get(key)
        t_m   = pair_tm   .get(key)

        if d is None or gamma is None or t_m is None:
            ax.text(0.5, 0.5, 'weights not loaded', ha='center', va='center',
                    transform=ax.transAxes)
            continue

        w_pair  = pair_weights.get(key)
        w_label = 'weights' if w_pair is not None else 'w=1'

        # 3+1D ground truth at t_m
        if key in sh_snaps:
            snaps = sh_snaps[key]
            k_sq, spec, t_act = interp_sh_at_t(snaps, t_m)
            k_sh, y_sh = normalize_sh(k_sq, spec, L, zero_pad)
            kR_sh = k_sh * d
            r_sh    = np.sqrt(k_sq[k_sq > 0])
            N_modes = np.maximum(1., 4 * np.pi * r_sh**2)
            yerr    = y_sh * np.sqrt(2. / N_modes)
            ax.plot(kR_sh, y_sh, color=color, lw=2, label=f'3+1D  (t={t_act:.1f})')
            ax.fill_between(kR_sh, np.maximum(y_sh - yerr, 1e-300), y_sh + yerr,
                            color=color, alpha=0.2, linewidth=0)

        # Direct 1D BM validation run at t_m
        if key in bm_direct:
            w_d, s_d, _ = bm_direct[key]
            ax.plot(w_d * d, s_d, color=color, lw=1.5,
                    linestyle=':', label='1+1D direct run')

        # Scan cumulative spectrum at t_m
        g_cl = float(np.clip(gamma, gammas_grid.min(), gammas_grid.max()))
        scan_at_tm = interp_fn(np.array([[g_cl, min(t_m, scan_t[-1])]]))[0]
        ax.plot(k_out * d, scan_at_tm, color=color, lw=1.5,
                linestyle='-.', label='1+1D scan @ t_m')

        # Reconstructed from scan increments up to t_m (includes t_m as endpoint)
        bm_spec = bm_reconstruct(gamma, w_pair, wt, t_m,
                                  interp_fn, scan_t, gammas_grid)
        ax.plot(k_out * d, bm_spec, color=color, lw=2,
                linestyle='--', label=f'1+1D scan recon ({w_label})')

        ax.set_title(rf'$\bar{{\lambda}}={lb}$,  pair {pair["idx"]},  $d={d:.2f}$')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(r'$k R_*$')
        ax.legend(frameon=False, fontsize=9)
        ax.set_ylim(bottom=1e-3)

    # ------------------------------------------------------------------ #
    # Panel 4: 3-bubble comparison at t_m_3b                             #
    # ------------------------------------------------------------------ #
    ax = axes[3]

    if (lb, '3b') in sh_snaps:
        snaps = sh_snaps[(lb, '3b')]
        k_sq, spec, t_act = interp_sh_at_t(snaps, t_m_3b)
        k_3b, y_3b = normalize_sh(k_sq, spec, L, zero_pad)
        r_3b    = np.sqrt(k_sq[k_sq > 0])
        N_modes = np.maximum(1., 4 * np.pi * r_3b**2)
        yerr    = y_3b * np.sqrt(2. / N_modes)
        ax.plot(k_3b, y_3b, color=colors_3b, lw=2,
                label=f'3+1D 3-bubble  (t={t_act:.1f})')
        ax.fill_between(k_3b, np.maximum(y_3b - yerr, 1e-300), y_3b + yerr,
                        color=colors_3b, alpha=0.2, linewidth=0)

    if lb in trio_weights:
        wd     = trio_weights[lb]
        w_arr  = wd['weights']
        pair_i = wd['pair_i']
        pair_j = wd['pair_j']
        gammas_trio = wd['gammas']

        combined_bm = np.zeros(len(k_out))
        combined_sh = np.zeros(len(k_out))

        for c_idx, (i, j) in enumerate(zip(pair_i, pair_j)):
            match = [p for p in case['pairs'] if set(p['idx']) == {i, j}]
            if not match:
                print(f'  WARNING: pair ({i},{j}) not in CASES for lb={lb}')
                continue
            pair_m  = match[0]
            w_trio  = w_arr[c_idx]
            gamma_p = float(gammas_trio[c_idx])

            combined_bm += bm_reconstruct(
                gamma_p, w_trio, wt, t_m_3b, interp_fn, scan_t, gammas_grid)
            if (lb, pair_m['idx']) in sh_snaps:
                combined_sh += sh_reconstruct(
                    lb, pair_m['idx'], w_trio, wt, t_m_3b, k_out, L, zero_pad)

        ax.plot(k_out, combined_bm, color='black', lw=2,
                linestyle='--', label='Recon: 1+1D scan × weights')
        ax.plot(k_out, combined_sh, color='gray', lw=2,
                linestyle=':', label='Recon: 3+1D pairs × weights')
    else:
        ax.text(0.5, 0.5, 'Run scripts/run_weights_N3.sh\nto enable reconstruction',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)

    ax.set_title(rf'$\bar{{\lambda}}={lb}$,  3-bubble  ($t_m={t_m_3b:.1f}$)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$k$ [physical]')
    ax.legend(frameon=False, fontsize=9)
    ax.set_ylim(bottom=1e2)

    axes[0].set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
    fig.suptitle(rf'N3 reconstruction — $\bar{{\lambda}}={lb}$', y=1.01)
    plt.tight_layout()
    plt.savefig(REPO_ROOT / f'n3_reconstruction_lb{lb}.pdf', bbox_inches='tight')
    plt.show()